# Prepare Historical Data

This notebook prepares historical climate and glacier mass balance data for the Mont Blanc study area before training the machine learning model.

The historical data includes:
- French Alps glacier mass balance from 1967–2015
- ERA5 monthly temperature from 1967–2015
- ERA5 monthly precipitation from 1967–2015

The glacier dataset will be filtered to glaciers located inside the geographic bounds of the Mont Blanc terrain model.

The final goal is to combine glacier and climate data into one yearly dataset that can be used to train and test the model.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Historical Data

The first step is to set the paths to the raw glacier and climate files and make sure each file can be found before working with the data.

In [2]:
# The notebook is inside data-science/notebooks,
# so move up one level to reach data-science.
DATA_DIR = Path("..") / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

mass_balance_file = RAW_DIR / "Bolibar2020_FrenchAlps_MassBalance_1967-2015.nc"
temperature_file = RAW_DIR / "ERA5_MontBlanc_Temperature_1967-2015.nc"
precipitation_file = RAW_DIR / "ERA5_MontBlanc_Precipitation_1967-2015.nc"

print("Mass balance:", mass_balance_file.exists())
print("Temperature:", temperature_file.exists())
print("Precipitation:", precipitation_file.exists())

Mass balance: True
Temperature: True
Precipitation: True


## Inspect Glacier Mass Balance Data

The French Alps glacier mass balance data contains the values the model will be trained to predict. Before using it, the dataset needs to be checked for the correct years, glacier records, missing values, and data types.

The dataset will later be filtered to glaciers located within the Mont Blanc terrain study area.

**Source:** Bolibar et al. (2020), reconstructed annual glacier-wide mass balance dataset for glaciers in the French Alps.

In [3]:
# Load the French Alps glacier mass balance dataset.
mass_balance = xr.open_dataset(mass_balance_file)

mass_balance

<xarray.Dataset> Size: 656kB
Dimensions:    (RGI_ID: 661, year: 49, GLIMS_ID: 661, name: 661)
Coordinates:
  * RGI_ID     (RGI_ID) int32 3kB 3323 0 3457 0 0 0 ... 0 3730 3731 3732 3733
  * year       (year) int32 196B 1967 1968 1969 1970 ... 2012 2013 2014 2015
  * GLIMS_ID   (GLIMS_ID) <U14 37kB 'G005930E45041N' ... 'G007170E45398N'
  * name       (name) <U37 98kB 'du Taillefer (Nord)' ... "des Sources de l'A...
Data variables:
    SMB        (RGI_ID, year) float64 259kB ...
    SMB_ASTER  (RGI_ID, year) float64 259kB ...
Attributes:
    Content:      French Alpine annual glacier-wide MB reconstructions (1967-...
    Author:       Jordi Bolibar
    Affiliation:  Institute of Environmental Geosciences (University Grenoble...

In [4]:
# Inspect the glacier coordinate information.
print("Coordinates:")
print(list(mass_balance.coords))

print("\nData variables:")
print(list(mass_balance.data_vars))

Coordinates:
['GLIMS_ID', 'name', 'RGI_ID', 'year']

Data variables:
['SMB', 'SMB_ASTER']


In [5]:
# Check the mass balance dataset for missing values and year coverage
print("Year range:", mass_balance["year"].min(), "-", mass_balance["year"].max())

print("\nMissing values:")
print(mass_balance.isnull().sum())

print("\nData types:")
print(mass_balance.dtypes)

Year range: <xarray.DataArray 'year' ()> Size: 4B
array(1967, dtype=int32) - <xarray.DataArray 'year' ()> Size: 4B
array(2015, dtype=int32)

Missing values:
<xarray.Dataset> Size: 16B
Dimensions:    ()
Data variables:
    SMB        int64 8B 2496
    SMB_ASTER  int64 8B 2496
Attributes:
    Content:      French Alpine annual glacier-wide MB reconstructions (1967-...
    Author:       Jordi Bolibar
    Affiliation:  Institute of Environmental Geosciences (University Grenoble...

Data types:
Frozen({'SMB': dtype('float64'), 'SMB_ASTER': dtype('float64')})


In [6]:
# Create a table containing each glacier record's ID, name, and location.
glacier_info = pd.DataFrame({
    "glacier_index": range(mass_balance.sizes["RGI_ID"]),
    "RGI_ID": mass_balance["RGI_ID"].values,
    "GLIMS_ID": mass_balance["GLIMS_ID"].values,
    "glacier_name": mass_balance["name"].values
})

# Extract longitude and latitude from the GLIMS ID.
glacier_info["longitude"] = (
    glacier_info["GLIMS_ID"].str[1:7].astype(float) / 1000
)

glacier_info["latitude"] = (
    glacier_info["GLIMS_ID"].str[8:13].astype(float) / 1000
)

glacier_info.head()

,glacier_index,RGI_ID,GLIMS_ID,glacier_name,longitude,latitude
0,0,3323,G005930E45041N,du Taillefer (Nord),5.930,45.041
1,1,0,G005936E45041N,des Dards 1,5.936,45.041
2,2,3457,G005937E44988N,du Grand Armet,5.937,44.988
3,3,0,G005940E45037N,des Dards 1,5.940,45.037
4,4,0,G005941E45033N,des Dards 1,5.941,45.033


In [7]:
# Geographic bounds of the Mont Blanc terrain model.
NORTH = 45.992668
SOUTH = 45.783327
WEST = 6.674194
EAST = 7.043610

# Keep only glaciers located inside the terrain model.
mont_blanc_glaciers = glacier_info[
    (glacier_info["latitude"] >= SOUTH) &
    (glacier_info["latitude"] <= NORTH) &
    (glacier_info["longitude"] >= WEST) &
    (glacier_info["longitude"] <= EAST)
].copy()

print("Glaciers in study area:", len(mont_blanc_glaciers))

mont_blanc_glaciers

Glaciers in study area: 58


,glacier_index,RGI_ID,GLIMS_ID,glacier_name,longitude,latitude
423,423,3344,G006776E45799N,de la Berangere,6.776,45.799
424,424,3601,G006778E45809N,d'Armancette,6.778,45.809
427,427,3650,G006783E45817N,de Covagnet,6.783,45.817
429,429,3651,G006784E45784N,de Tre la Tete 1,6.784,45.784
431,431,3276,G006785E45822N,du Miage 1,6.785,45.822
437,437,3388,G006792E45817N,du Miage 2,6.792,45.817
447,447,3579,G006799E45819N,du Miage 3,6.799,45.819
448,448,3649,G006799E45827N,du Miage 4,6.799,45.827
449,449,3296,G006799E45837N,du Tricot,6.799,45.837
459,459,3648,G006811E45849N,de Bionnassay,6.811,45.849


In [8]:
# Convert SMB data to a glacier-year table using the unique glacier index.
smb_df = pd.DataFrame({
    "glacier_index": (
        pd.Series(range(mass_balance.sizes["RGI_ID"]))
        .repeat(mass_balance.sizes["year"])
        .values
    ),
    "year": list(mass_balance["year"].values) * mass_balance.sizes["RGI_ID"],
    "mass_balance": mass_balance["SMB"].values.flatten()
})

smb_df.head()

,glacier_index,year,mass_balance
0,0,1967,-0.569014
1,0,1968,-0.740084
2,0,1969,-0.670044
3,0,1970,-1.076898
4,0,1971,-1.046018


In [10]:
# Keep only SMB records for glaciers inside the Mont Blanc study area.
mont_blanc_smb = smb_df.merge(
    mont_blanc_glaciers,
    on="glacier_index",
    how="inner"
)

print("Rows:", len(mont_blanc_smb))
print("Glacier records:", mont_blanc_smb["glacier_index"].nunique())
print("Years:", mont_blanc_smb["year"].min(), "-", mont_blanc_smb["year"].max())

mont_blanc_smb.head()

Rows: 2842
Glacier records: 58
Years: 1967 - 2015


,glacier_index,year,mass_balance,RGI_ID,GLIMS_ID,glacier_name,longitude,latitude
0,423,1967,-1.019276,3344,G006776E45799N,de la Berangere,6.776,45.799
1,423,1968,1.020175,3344,G006776E45799N,de la Berangere,6.776,45.799
2,423,1969,-1.551294,3344,G006776E45799N,de la Berangere,6.776,45.799
3,423,1970,-0.300338,3344,G006776E45799N,de la Berangere,6.776,45.799
4,423,1971,-0.546125,3344,G006776E45799N,de la Berangere,6.776,45.799


In [11]:
# Check the Mont Blanc SMB data for missing values.
print("Missing mass balance values:", mont_blanc_smb["mass_balance"].isna().sum())

print("\nMass balance summary:")
print(mont_blanc_smb["mass_balance"].describe())

Missing mass balance values: 60

Mass balance summary:
count    2782.000000
mean       -0.710393
std         0.967558
min        -3.639968
25%        -1.402411
50%        -0.710024
75%        -0.069902
max         2.359916
Name: mass_balance, dtype: float64


In [12]:
# Verify that the missing SMB values already exist in the original NetCDF.
study_indices = mont_blanc_glaciers["glacier_index"].values

raw_study_smb = mass_balance["SMB"].values[study_indices, :]

print("Missing SMB values in original NetCDF:", pd.isna(raw_study_smb).sum())
print("Missing SMB values after reshaping:", mont_blanc_smb["mass_balance"].isna().sum())

Missing SMB values in original NetCDF: 60
Missing SMB values after reshaping: 60


In [13]:
# Remove glacier-year records that do not have an SMB target value.
mont_blanc_smb = mont_blanc_smb.dropna(
    subset=["mass_balance"]
).copy()

print("Usable SMB records:", len(mont_blanc_smb))

Usable SMB records: 2782


## Inspect Historical Temperature Data

The ERA5 temperature file contains monthly 2-meter temperature data for the Mont Blanc region from 1967–2015. The file structure, dates, coordinates, and units need to be checked before processing the values.

**Source:** ERA5 historical climate data from the Copernicus Climate Data Store.

In [14]:
# Load the ERA5 monthly climate datasets.
temperature = xr.open_dataset(temperature_file)
precipitation = xr.open_dataset(precipitation_file)

print("Temperature variables:", list(temperature.data_vars))
print("Temperature longitudes:", temperature["longitude"].values)
print("Temperature latitudes:", temperature["latitude"].values)

print("\nPrecipitation variables:", list(precipitation.data_vars))
print("Precipitation longitudes:", precipitation["longitude"].values)
print("Precipitation latitudes:", precipitation["latitude"].values)

Temperature variables: ['t2m']
Temperature longitudes: [6.5  6.75 7.   7.25]
Temperature latitudes: [46.   45.75]

Precipitation variables: ['tp']
Precipitation longitudes: [6.5  6.75 7.   7.25]
Precipitation latitudes: [46.   45.75]


## Process Historical Temperature Data

ERA5 monthly 2 m temperature data is used to create yearly climate features for each glacier in the Mont Blanc study area. Temperature values are converted from Kelvin to Celsius before calculating annual and seasonal features.

In [15]:
# Convert ERA5 temperature from Kelvin to Celsius.
temperature_c = temperature["t2m"] - 273.15

print("Missing temperature values:", temperature_c.isnull().sum().item())
print("Minimum temperature:", float(temperature_c.min()))
print("Maximum temperature:", float(temperature_c.max()))

Missing temperature values: 0
Minimum temperature: -15.701507568359375
Maximum temperature: 18.856842041015625


In [16]:
# Assign each month to a hydrological year.
# October-December belong to the following year.
hydrological_year = xr.where(
    temperature_c["valid_time"].dt.month >= 10,
    temperature_c["valid_time"].dt.year + 1,
    temperature_c["valid_time"].dt.year
)

temperature_c = temperature_c.assign_coords(
    hydrological_year=("valid_time", hydrological_year.data)
)

# Calculate mean temperature for each October-September hydrological year.
annual_temperature = (
    temperature_c
    .groupby("hydrological_year")
    .mean(dim="valid_time")
)

annual_temperature

# Keep only complete October-September hydrological years.
annual_temperature = annual_temperature.sel(
    hydrological_year=slice(1968, 2015)
)

print(
    "Hydrological years:",
    int(annual_temperature.hydrological_year.min()),
    "-",
    int(annual_temperature.hydrological_year.max())
)

print("Number of complete years:", annual_temperature.sizes["hydrological_year"])
print("Shape:", annual_temperature.shape)

# Winter temperature: October-March.
winter_temperature = (
    temperature_c
    .where(
        temperature_c["valid_time"].dt.month.isin([10, 11, 12, 1, 2, 3]),
        drop=True
    )
    .groupby("hydrological_year")
    .mean(dim="valid_time")
    .sel(hydrological_year=slice(1968, 2015))
)

# Summer temperature: April-September.
summer_temperature = (
    temperature_c
    .where(
        temperature_c["valid_time"].dt.month.isin([4, 5, 6, 7, 8, 9]),
        drop=True
    )
    .groupby("hydrological_year")
    .mean(dim="valid_time")
    .sel(hydrological_year=slice(1968, 2015))
)

print("Winter shape:", winter_temperature.shape)
print("Summer shape:", summer_temperature.shape)

Hydrological years: 1968 - 2015
Number of complete years: 48
Shape: (48, 2, 4)
Winter shape: (48, 2, 4)
Summer shape: (48, 2, 4)


## Process Historical Precipitation Data

ERA5 total precipitation is converted from meters per day to monthly totals in millimeters. Annual, winter, and summer precipitation features are then calculated using the same October–September hydrological-year structure as the temperature features.

In [17]:
# Convert ERA5 total precipitation from meters per day
# to monthly precipitation totals in millimeters.
days_in_month = precipitation["valid_time"].dt.days_in_month

precipitation_mm = (
    precipitation["tp"]
    * 1000
    * days_in_month
)

print("Missing precipitation values:", precipitation_mm.isnull().sum().item())
print("Minimum monthly precipitation (mm):", float(precipitation_mm.min()))
print("Maximum monthly precipitation (mm):", float(precipitation_mm.max()))

# Assign each precipitation month to a hydrological year.
# October-December belong to the following year.
precip_hydrological_year = xr.where(
    precipitation_mm["valid_time"].dt.month >= 10,
    precipitation_mm["valid_time"].dt.year + 1,
    precipitation_mm["valid_time"].dt.year
)

precipitation_mm = precipitation_mm.assign_coords(
    hydrological_year=("valid_time", precip_hydrological_year.data)
)

# Calculate total precipitation for each complete October-September year.
annual_precipitation = (
    precipitation_mm
    .groupby("hydrological_year")
    .sum(dim="valid_time")
    .sel(hydrological_year=slice(1968, 2015))
)

print(
    "Hydrological years:",
    int(annual_precipitation.hydrological_year.min()),
    "-",
    int(annual_precipitation.hydrological_year.max())
)

print("Number of complete years:", annual_precipitation.sizes["hydrological_year"])
print("Shape:", annual_precipitation.shape)

Missing precipitation values: 0
Minimum monthly precipitation (mm): 5.942344665527344
Maximum monthly precipitation (mm): 535.3431701660156
Hydrological years: 1968 - 2015
Number of complete years: 48
Shape: (48, 2, 4)


In [18]:
# Winter precipitation: October-March.
winter_precipitation = (
    precipitation_mm
    .where(
        precipitation_mm["valid_time"].dt.month.isin([10, 11, 12, 1, 2, 3]),
        drop=True
    )
    .groupby("hydrological_year")
    .sum(dim="valid_time")
    .sel(hydrological_year=slice(1968, 2015))
)

# Summer precipitation: April-September.
summer_precipitation = (
    precipitation_mm
    .where(
        precipitation_mm["valid_time"].dt.month.isin([4, 5, 6, 7, 8, 9]),
        drop=True
    )
    .groupby("hydrological_year")
    .sum(dim="valid_time")
    .sel(hydrological_year=slice(1968, 2015))
)

print("Winter shape:", winter_precipitation.shape)
print("Summer shape:", summer_precipitation.shape)

Winter shape: (48, 2, 4)
Summer shape: (48, 2, 4)


In [19]:
# Verify that the seasonal precipitation totals reconstruct the annual total.
precip_difference = (
    annual_precipitation
    - (winter_precipitation + summer_precipitation)
)

print(
    "Maximum annual precipitation difference:",
    float(abs(precip_difference).max())
)

print(
    "Temperature years match:",
    np.array_equal(
        annual_temperature["hydrological_year"].values,
        winter_temperature["hydrological_year"].values
    )
    and np.array_equal(
        annual_temperature["hydrological_year"].values,
        summer_temperature["hydrological_year"].values
    )
)

print(
    "Precipitation years match:",
    np.array_equal(
        annual_precipitation["hydrological_year"].values,
        winter_precipitation["hydrological_year"].values
    )
    and np.array_equal(
        annual_precipitation["hydrological_year"].values,
        summer_precipitation["hydrological_year"].values
    )
)

print(
    "Temperature and precipitation years match:",
    np.array_equal(
        annual_temperature["hydrological_year"].values,
        annual_precipitation["hydrological_year"].values
    )
)

Maximum annual precipitation difference: 0.0
Temperature years match: True
Precipitation years match: True
Temperature and precipitation years match: True


## Assign Climate Features to Glacier Locations

The ERA5 climate features are spatially interpolated to each glacier record using its latitude and longitude. This preserves spatial differences across the Mont Blanc study area rather than assigning one regional climate average to every glacier.

In [20]:
# Sort latitude so xarray interpolation works cleanly.
annual_temperature_interp = annual_temperature.sortby("latitude")
winter_temperature_interp = winter_temperature.sortby("latitude")
summer_temperature_interp = summer_temperature.sortby("latitude")

annual_precipitation_interp = annual_precipitation.sortby("latitude")
winter_precipitation_interp = winter_precipitation.sortby("latitude")
summer_precipitation_interp = summer_precipitation.sortby("latitude")

# Convert glacier coordinates to xarray DataArrays for vectorized interpolation.
glacier_latitudes = xr.DataArray(
    mont_blanc_glaciers["latitude"].values,
    dims="glacier_index"
)

glacier_longitudes = xr.DataArray(
    mont_blanc_glaciers["longitude"].values,
    dims="glacier_index"
)

# Interpolate each climate feature to the glacier locations.
glacier_annual_temperature = annual_temperature_interp.interp(
    latitude=glacier_latitudes,
    longitude=glacier_longitudes
)

glacier_winter_temperature = winter_temperature_interp.interp(
    latitude=glacier_latitudes,
    longitude=glacier_longitudes
)

glacier_summer_temperature = summer_temperature_interp.interp(
    latitude=glacier_latitudes,
    longitude=glacier_longitudes
)

glacier_annual_precipitation = annual_precipitation_interp.interp(
    latitude=glacier_latitudes,
    longitude=glacier_longitudes
)

glacier_winter_precipitation = winter_precipitation_interp.interp(
    latitude=glacier_latitudes,
    longitude=glacier_longitudes
)

glacier_summer_precipitation = summer_precipitation_interp.interp(
    latitude=glacier_latitudes,
    longitude=glacier_longitudes
)

print("Annual temperature shape:", glacier_annual_temperature.shape)
print("Winter temperature shape:", glacier_winter_temperature.shape)
print("Summer temperature shape:", glacier_summer_temperature.shape)
print("Annual precipitation shape:", glacier_annual_precipitation.shape)
print("Winter precipitation shape:", glacier_winter_precipitation.shape)
print("Summer precipitation shape:", glacier_summer_precipitation.shape)

Annual temperature shape: (48, 58)
Winter temperature shape: (48, 58)
Summer temperature shape: (48, 58)
Annual precipitation shape: (48, 58)
Winter precipitation shape: (48, 58)
Summer precipitation shape: (48, 58)


In [21]:
climate_features = pd.DataFrame({
    "hydrological_year": np.repeat(
        glacier_annual_temperature["hydrological_year"].values,
        glacier_annual_temperature.sizes["glacier_index"]
    ),

    "glacier_index": np.tile(
        mont_blanc_glaciers["glacier_index"].values,
        glacier_annual_temperature.sizes["hydrological_year"]
    ),

    "annual_temperature_c": glacier_annual_temperature.values.flatten(),
    "winter_temperature_c": glacier_winter_temperature.values.flatten(),
    "summer_temperature_c": glacier_summer_temperature.values.flatten(),

    "annual_precipitation_mm": glacier_annual_precipitation.values.flatten(),
    "winter_precipitation_mm": glacier_winter_precipitation.values.flatten(),
    "summer_precipitation_mm": glacier_summer_precipitation.values.flatten()
})

print(climate_features.shape)
print(climate_features.head())
print(climate_features["hydrological_year"].min(),
      climate_features["hydrological_year"].max())
print(climate_features["glacier_index"].nunique())

(2784, 8)
   hydrological_year  glacier_index  annual_temperature_c  \
0               1968            423              1.040538   
1               1968            424              1.064644   
2               1968            427              1.069756   
3               1968            429              0.959309   
4               1968            431              1.077550   

   winter_temperature_c  summer_temperature_c  annual_precipitation_mm  \
0             -4.237914              6.318990              1622.972595   
1             -4.219128              6.348415              1633.635529   
2             -4.218176              6.357689              1638.001434   
3             -4.310709              6.229326              1593.901842   
4             -4.213031              6.368132              1642.061705   

   winter_precipitation_mm  summer_precipitation_mm  
0               787.011292               835.961303  
1               790.549427               843.086102  
2               

In [22]:
# Add glacier metadata to each glacier-year climate record.
climate_features = climate_features.merge(
    mont_blanc_glaciers[
        [
            "glacier_index",
            "RGI_ID",
            "GLIMS_ID",
            "glacier_name",
            "latitude",
            "longitude"
        ]
    ],
    on="glacier_index",
    how="left"
)

print("Shape:", climate_features.shape)
print("Unique glacier records:", climate_features["glacier_index"].nunique())

print("\nMissing metadata:")
print(
    climate_features[
        ["RGI_ID", "GLIMS_ID", "glacier_name", "latitude", "longitude"]
    ].isnull().sum()
)

climate_features.head()

Shape: (2784, 13)
Unique glacier records: 58

Missing metadata:
RGI_ID          0
GLIMS_ID        0
glacier_name    0
latitude        0
longitude       0
dtype: int64


,hydrological_year,glacier_index,annual_temperature_c,winter_temperature_c,summer_temperature_c,annual_precipitation_mm,winter_precipitation_mm,summer_precipitation_mm,RGI_ID,GLIMS_ID,glacier_name,latitude,longitude
0,1968,423,1.040538,-4.237914,6.318990,1622.972595,787.011292,835.961303,3344,G006776E45799N,de la Berangere,45.799,6.776
1,1968,424,1.064644,-4.219128,6.348415,1633.635529,790.549427,843.086102,3601,G006778E45809N,d'Armancette,45.809,6.778
2,1968,427,1.069756,-4.218176,6.357689,1638.001434,791.076965,846.924469,3650,G006783E45817N,de Covagnet,45.817,6.783
3,1968,429,0.959309,-4.310709,6.229326,1593.901842,774.321153,819.580689,3651,G006784E45784N,de Tre la Tete 1,45.784,6.784
4,1968,431,1.077550,-4.213031,6.368132,1642.061705,792.160762,849.900943,3276,G006785E45822N,du Miage 1,45.822,6.785


In [23]:
# Prepare SMB target for merge.
smb_target = mont_blanc_smb[
    [
        "glacier_index",
        "year",
        "mass_balance"
    ]
].copy()

smb_target = smb_target.rename(
    columns={"year": "hydrological_year"}
)

# Merge SMB onto the climate features.
model_data = climate_features.merge(
    smb_target,
    on=["glacier_index", "hydrological_year"],
    how="inner"
)

print("Shape:", model_data.shape)
print("Years:", model_data["hydrological_year"].min(),
      "-", model_data["hydrological_year"].max())
print("Unique glacier records:", model_data["glacier_index"].nunique())
print("Missing mass balance:", model_data["mass_balance"].isnull().sum())

model_data.head()

Shape: (2724, 14)
Years: 1968 - 2015
Unique glacier records: 58
Missing mass balance: 0


,hydrological_year,glacier_index,annual_temperature_c,winter_temperature_c,summer_temperature_c,annual_precipitation_mm,winter_precipitation_mm,summer_precipitation_mm,RGI_ID,GLIMS_ID,glacier_name,latitude,longitude,mass_balance
0,1968,423,1.040538,-4.237914,6.318990,1622.972595,787.011292,835.961303,3344,G006776E45799N,de la Berangere,45.799,6.776,1.020175
1,1968,424,1.064644,-4.219128,6.348415,1633.635529,790.549427,843.086102,3601,G006778E45809N,d'Armancette,45.809,6.778,0.992590
2,1968,427,1.069756,-4.218176,6.357689,1638.001434,791.076965,846.924469,3650,G006783E45817N,de Covagnet,45.817,6.783,1.047586
3,1968,429,0.959309,-4.310709,6.229326,1593.901842,774.321153,819.580689,3651,G006784E45784N,de Tre la Tete 1,45.784,6.784,-0.154915
4,1968,431,1.077550,-4.213031,6.368132,1642.061705,792.160762,849.900943,3276,G006785E45822N,du Miage 1,45.822,6.785,-0.445979


In [24]:
# Final integrity checks before saving the historical modeling dataset.

print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

print(
    "Duplicate glacier-year records:",
    model_data.duplicated(
        subset=["glacier_index", "hydrological_year"]
    ).sum()
)

print("\nMissing values:")
print(model_data.isnull().sum())

print(
    "\nInfinite climate values:",
    np.isinf(
        model_data[
            [
                "annual_temperature_c",
                "winter_temperature_c",
                "summer_temperature_c",
                "annual_precipitation_mm",
                "winter_precipitation_mm",
                "summer_precipitation_mm"
            ]
        ]
    ).sum().sum()
)

print(
    "\nMass balance range:",
    model_data["mass_balance"].min(),
    "to",
    model_data["mass_balance"].max()
)

Rows: 2724
Columns: 14
Duplicate glacier-year records: 0

Missing values:
hydrological_year          0
glacier_index              0
annual_temperature_c       0
winter_temperature_c       0
summer_temperature_c       0
annual_precipitation_mm    0
winter_precipitation_mm    0
summer_precipitation_mm    0
RGI_ID                     0
GLIMS_ID                   0
glacier_name               0
latitude                   0
longitude                  0
mass_balance               0
dtype: int64

Infinite climate values: 0

Mass balance range: -3.6399684 to 2.359916


In [25]:
# Save the prepared historical modeling dataset.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_DIR / "MontBlanc_Historical_Model_Data_1968-2015.csv"

model_data.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

Saved: ../data/processed/MontBlanc_Historical_Model_Data_1968-2015.csv
Rows: 2724
Columns: 14
